In [108]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from datetime import date

# Hey clientes dataset

In [109]:
# Load the dataset
df = pd.read_csv('../../Data/dataset_transacciones/hey_clientes.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15025 entries, 0 to 15024
Data columns (total 22 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   user_id                  15025 non-null  object 
 1   edad                     15025 non-null  int64  
 2   sexo                     15025 non-null  object 
 3   estado                   14593 non-null  object 
 4   ciudad                   14593 non-null  object 
 5   nivel_educativo          15025 non-null  object 
 6   ocupacion                15025 non-null  object 
 7   ingreso_mensual_mxn      15025 non-null  int64  
 8   antiguedad_dias          15025 non-null  int64  
 9   es_hey_pro               15025 non-null  bool   
 10  nomina_domiciliada       15025 non-null  bool   
 11  canal_apertura           15025 non-null  object 
 12  score_buro               15025 non-null  int64  
 13  dias_desde_ultimo_login  15025 non-null  int64  
 14  preferencia_canal     

## 1. Age Band Feature

Bins de `edad`: [18-25, 26-35, 36-50, 51+]

In [110]:
def create_age_band(edad):
    if 18 <= edad <= 25:
        return '18-25'
    elif 26 <= edad <= 35:
        return '26-35'
    elif 36 <= edad <= 50:
        return '36-50'
    elif edad >= 51:
        return '51+'
    else:
        return None

df['age_band'] = df['edad'].apply(create_age_band)
print(df[['edad', 'age_band']].head())

   edad age_band
0    21    18-25
1    18    18-25
2    23    18-25
3    32    26-35
4    26    26-35


## 2. Income Tier Feature
Quantile-bin `ingreso_mensual_mx` en tiers Q1-Q4.

In [111]:
df['income_tier'] = pd.qcut(df['ingreso_mensual_mxn'], q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'])
print(df[['ingreso_mensual_mxn', 'income_tier']].head())

   ingreso_mensual_mxn income_tier
0                24500          Q3
1                19000          Q2
2                14000          Q1
3                61000          Q4
4                27000          Q3


## 3. Engagement Score Feature
Normalizar y combinar: `dias_desde_ultimo_login`, `satisfaccion_1_10`, `es_hey_pro`, `nomina_domiciliada`.

In [112]:
# Normalize components
scaler = MinMaxScaler()

# Inverse of days since last login (higher frequency = higher engagement)
df['login_freq'] = 1 / (df['dias_desde_ultimo_login'] + 1)  # +1 to avoid division by zero
df['login_freq_norm'] = scaler.fit_transform(df[['login_freq']])

# Normalize satisfaction (already 1-10, but scale to 0-1)
df['satisfaccion_norm'] = (df['satisfaccion_1_10'] - 1) / 9  # Min 1, max 10

# Binary features to 0-1
df['es_hey_pro_num'] = df['es_hey_pro'].astype(int)
df['nomina_domiciliada_num'] = df['nomina_domiciliada'].astype(int)

# Combine into engagement score (average of normalized components)
df['engagement_score'] = (
    df['login_freq_norm'] + 
    df['satisfaccion_norm'] + 
    df['es_hey_pro_num'] + 
    df['nomina_domiciliada_num']
) / 4

print(df[['dias_desde_ultimo_login', 'satisfaccion_1_10', 'es_hey_pro', 'nomina_domiciliada', 'engagement_score']].head())

   dias_desde_ultimo_login  satisfaccion_1_10  es_hey_pro  nomina_domiciliada  \
0                        1               10.0        True               False   
1                        3                8.0        True               False   
2                        3                8.0        True               False   
3                       16               10.0       False               False   
4                        1                7.0        True               False   

   engagement_score  
0          0.624306  
1          0.505903  
2          0.505903  
3          0.263399  
4          0.540972  


## 4. Credit Health Feature
La neta hay que quedarse con el score buro

## 5. Digital Maturity Feature
Combinar `canal_apertura`, `preferencia_canal`, y la inversa de `dias_desde_ultimo_login`.

In [113]:
# Map canal_apertura to score (App = 1, Fan Shop = 0)
df['canal_score'] = df['canal_apertura'].map({'App': 1, 'Fan Shop': 0})

# Map preferencia_canal to scores (assuming app_ios/android/huawei are similar)
df['preferencia_score'] = df['preferencia_canal'].map({
    'app_ios': 1, 
    'app_android': 1, 
    'app_huawei': 1
}).fillna(0)  # If other values, assume 0

# Use the same login_freq_norm from engagement score

# Combine into digital maturity score
df['digital_maturity'] = (
    df['canal_score'] + 
    df['preferencia_score'] + 
    df['login_freq_norm']
) / 3

print(df[['canal_apertura', 'preferencia_canal', 'dias_desde_ultimo_login', 'digital_maturity']].head())

  canal_apertura preferencia_canal  dias_desde_ultimo_login  digital_maturity
0            App       app_android                        1          0.832407
1            App       app_android                        3          0.748611
2            App           app_ios                        3          0.748611
3       Fan Shop           app_ios                       16          0.351198
4       Fan Shop           app_ios                        1          0.499074


## 6. Tenure Band Feature
Mejor quedarse con los días de antiguedad

## 7. Vulnerability Flag Feature
Flag basada en `recibe_remesas`, `ocupacion==Desempleado`, y un bajo estrato de ingreso (Q1).

In [114]:
df['vulnerability_flag'] = (
    df['recibe_remesas'] | 
    (df['ocupacion'] == 'Desempleado') | 
    (df['income_tier'] == 'Q1')
).astype(int)

print(df[['recibe_remesas', 'ocupacion', 'income_tier', 'vulnerability_flag']].head())

   recibe_remesas   ocupacion income_tier  vulnerability_flag
0           False    Empleado          Q3                   0
1           False  Estudiante          Q2                   0
2           False  Estudiante          Q1                   1
3            True    Empleado          Q4                   1
4           False  Empresario          Q3                   0


## 8. Financial Sophistication Feature
Combinar `nivel_educativo`, `usa_hey_shop` (Como un proxy de la actividad de inversion), y `score_buro`.

In [115]:
# Map education level to scores
education_scores = {
    'Secundaria': 1,
    'Preparatoria': 2,
    'Licenciatura': 3,
    'Posgrado': 4
}
df['education_score'] = df['nivel_educativo'].map(education_scores)
df['education_score_norm'] = scaler.fit_transform(df[['education_score']])

# Investment proxy (usa_hey_shop as binary)
df['investment_proxy'] = df['usa_hey_shop'].astype(int)

df['score_buro_norm'] = scaler.fit_transform(df[['score_buro']])
# Combine with normalized score_buro
df['financial_sophistication'] = (
    df['education_score_norm'] + 
    df['investment_proxy'] + 
    df['score_buro_norm']
) / 3

print(df[['nivel_educativo', 'usa_hey_shop', 'score_buro', 'financial_sophistication']].head())

  nivel_educativo  usa_hey_shop  score_buro  financial_sophistication
0    Preparatoria          True         527                  0.583784
1    Preparatoria          True         714                  0.696096
2    Licenciatura          True         454                  0.651051
3        Posgrado         False         837                  0.658859
4    Preparatoria          True         533                  0.587387


## Final clients dataframe

In [116]:
clients_final = df[['user_id', 'age_band','income_tier', 'es_hey_pro', 'engagement_score', 'digital_maturity', 'vulnerability_flag', 'financial_sophistication']]

# Hey products dataset

In [117]:
df = pd.read_csv('../../Data/dataset_transacciones/hey_productos.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38909 entries, 0 to 38908
Data columns (total 13 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   producto_id              38909 non-null  object 
 1   user_id                  38909 non-null  object 
 2   tipo_producto            38909 non-null  object 
 3   fecha_apertura           38909 non-null  object 
 4   estatus                  38909 non-null  object 
 5   limite_credito           14317 non-null  float64
 6   saldo_actual             35159 non-null  float64
 7   utilizacion_pct          14317 non-null  float64
 8   tasa_interes_anual       18791 non-null  float64
 9   plazo_meses              4550 non-null   float64
 10  monto_mensualidad        4550 non-null   float64
 11  fecha_ultimo_movimiento  38909 non-null  object 
 12  es_dato_sintetico        38909 non-null  bool   
dtypes: bool(1), float64(6), object(6)
memory usage: 3.6+ MB


In [118]:
# Pre-process data
# Convert date columns
df['fecha_apertura'] = pd.to_datetime(df['fecha_apertura'])
df['fecha_ultimo_movimiento'] = pd.to_datetime(df['fecha_ultimo_movimiento'])

# Define constants
CREDIT_PRODUCTS = ['tarjeta_credito_hey', 'tarjeta_credito_garantizada', 'tarjeta_credito_negocios', 
                   'credito_personal', 'credito_auto', 'credito_nomina']
HAS_CREDIT_PRODUCTS = ['tarjeta_credito_hey', 'credito_personal', 'credito_auto', 'credito_nomina']
INSURANCE_PRODUCTS = ['seguro_vida', 'seguro_compras']
CURRENT_DATE = date(2026, 4, 25)

print("Data preprocessing complete")

Data preprocessing complete


In [119]:
# Group Data by user_id
grouped = df.groupby('user_id')
print(f"Number of unique users: {len(grouped)}")

Number of unique users: 15025


## 1. Product diversity score

In [120]:
# Compute product_diversity_score
product_diversity_score = grouped['tipo_producto'].nunique()
print("product_diversity_score computed")
print(product_diversity_score.head())

product_diversity_score computed
user_id
USR-00001    2
USR-00002    2
USR-00003    2
USR-00004    3
USR-00005    2
Name: tipo_producto, dtype: int64


## 2. Has credit

In [121]:
# Compute has_credit
def has_credit(products):
    return any(p in HAS_CREDIT_PRODUCTS for p in products)

has_credit_flag = grouped['tipo_producto'].apply(has_credit)
print("has_credit computed")
print(has_credit_flag.head())

has_credit computed
user_id
USR-00001    True
USR-00002    True
USR-00003    True
USR-00004    True
USR-00005    True
Name: tipo_producto, dtype: bool


## 3. Average credit utilization

In [122]:
# Compute avg_credit_utilization
def avg_credit_util(group):
    credit_rows = group[group['tipo_producto'].isin(CREDIT_PRODUCTS)]
    if not credit_rows.empty:
        return credit_rows['utilizacion_pct'].mean()
    return np.nan

avg_credit_utilization = grouped.apply(avg_credit_util)
print("avg_credit_utilization computed")
print(avg_credit_utilization.head())

avg_credit_utilization computed
user_id
USR-00001    0.6166
USR-00002    0.2783
USR-00003    0.3091
USR-00004    0.3538
USR-00005    0.6326
dtype: float64


/tmp/ipykernel_18389/2563412257.py:8: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  avg_credit_utilization = grouped.apply(avg_credit_util)


## 4. Max credit limit

In [123]:
# Compute max_credit_limit
max_credit_limit = grouped['limite_credito'].max()
print("max_credit_limit computed")
print(max_credit_limit.head())

max_credit_limit computed
user_id
USR-00001    144000.0
USR-00002     22000.0
USR-00003      5000.0
USR-00004     67000.0
USR-00005    136000.0
Name: limite_credito, dtype: float64


## 5. Has investment

In [124]:
# Compute has_investment
def has_investment(group):
    inv_rows = group[(group['tipo_producto'] == 'inversion_hey') & (group['saldo_actual'] > 0)]
    return not inv_rows.empty

has_investment_flag = grouped.apply(has_investment)
print("has_investment computed")
print(has_investment_flag.head())

has_investment computed
user_id
USR-00001    False
USR-00002    False
USR-00003    False
USR-00004     True
USR-00005    False
dtype: bool


/tmp/ipykernel_18389/4240445375.py:6: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  has_investment_flag = grouped.apply(has_investment)


## 6. Has insurance

In [125]:
# Compute has_insurance
def has_insurance(products):
    return any(p in INSURANCE_PRODUCTS for p in products)

has_insurance_flag = grouped['tipo_producto'].apply(has_insurance)
print("has_insurance computed")
print(has_insurance_flag.head())

has_insurance computed
user_id
USR-00001    False
USR-00002    False
USR-00003    False
USR-00004    False
USR-00005    False
Name: tipo_producto, dtype: bool


## 7. Secured card flag

In [126]:
# Compute secured_card_flag
def has_secured_card(products):
    return 'tarjeta_credito_garantizada' in products

secured_card_flag = grouped['tipo_producto'].apply(has_secured_card)
print("secured_card_flag computed")
print(secured_card_flag.head())

secured_card_flag computed
user_id
USR-00001    False
USR-00002    False
USR-00003    False
USR-00004    False
USR-00005    False
Name: tipo_producto, dtype: bool


## 8. Portfolio_age_days

In [127]:
# Compute portfolio_age_days
portfolio_age_days = grouped['fecha_apertura'].min().apply(lambda x: (CURRENT_DATE - x.date()).days)
print("portfolio_age_days computed")
print(portfolio_age_days.head())

portfolio_age_days computed
user_id
USR-00001    1287
USR-00002     387
USR-00003    1126
USR-00004    1118
USR-00005     746
Name: fecha_apertura, dtype: int64


## 9. Credit product count

In [128]:
# Compute credit_product_count
def count_credit_products(products):
    return sum(1 for p in products if p in CREDIT_PRODUCTS)

credit_product_count = grouped['tipo_producto'].apply(count_credit_products)
print("credit_product_count computed")
print(credit_product_count.head())

credit_product_count computed
user_id
USR-00001    1
USR-00002    1
USR-00003    1
USR-00004    1
USR-00005    1
Name: tipo_producto, dtype: int64


## 10. Investment balance

In [129]:
# Compute investment_balance
def sum_investment_balance(group):
    inv_rows = group[group['tipo_producto'] == 'inversion_hey']
    return inv_rows['saldo_actual'].sum() if not inv_rows.empty else 0

investment_balance = grouped.apply(sum_investment_balance)
print("investment_balance computed")
print(investment_balance.head())

investment_balance computed
user_id
USR-00001         0.00
USR-00002         0.00
USR-00003         0.00
USR-00004    319425.53
USR-00005         0.00
dtype: float64


/tmp/ipykernel_18389/166588707.py:6: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  investment_balance = grouped.apply(sum_investment_balance)


## Hey product dataframe

In [130]:
# Assemble Final DataFrame
products_final = pd.DataFrame({
    'user_id': product_diversity_score.index,
    'product_diversity_score': product_diversity_score.values,
    'has_credit': has_credit_flag.values,
    'avg_credit_utilization': avg_credit_utilization.values,
    'max_credit_limit': max_credit_limit.values,
    'has_investment': has_investment_flag.values,
    'has_insurance': has_insurance_flag.values,
    'secured_card_flag': secured_card_flag.values,
    'portfolio_age_days': portfolio_age_days.values,
    'credit_product_count': credit_product_count.values,
    'investment_balance': investment_balance.values
})

print("Final features DataFrame created")

Final features DataFrame created


# Hey transactions dataset

In [131]:
df = pd.read_csv("../../Data/dataset_transacciones/hey_transacciones.csv",parse_dates=['fecha_hora'])
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 802384 entries, 0 to 802383
Data columns (total 22 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   transaccion_id       802384 non-null  object        
 1   user_id              802384 non-null  object        
 2   producto_id          802384 non-null  object        
 3   fecha_hora           802384 non-null  datetime64[ns]
 4   tipo_operacion       802384 non-null  object        
 5   canal                802384 non-null  object        
 6   monto                802384 non-null  float64       
 7   comercio_nombre      431846 non-null  object        
 8   categoria_mcc        802384 non-null  object        
 9   ciudad_transaccion   783117 non-null  object        
 10  estatus              802384 non-null  object        
 11  motivo_no_procesada  26609 non-null   object        
 12  intento_numero       802384 non-null  int64         
 13  meses_diferido

In [132]:
# Set reference date for recency as max fecha_hora
ref_date = df['fecha_hora'].max()
print(f"Reference date for recency: {ref_date}")

# Handle nulls: fill numeric nulls with 0, categorical with empty string if needed
numeric_cols = ['monto', 'cashback_generado', 'meses_diferidos']
df[numeric_cols] = df[numeric_cols].fillna(0)
df['motivo_no_procesada'] = df['motivo_no_procesada'].fillna('')
df['comercio_nombre'] = df['comercio_nombre'].fillna('')
df['dispositivo'] = df['dispositivo'].fillna('')

Reference date for recency: 2025-12-15 13:59:36


## Volume & Frequency Features

In [133]:
# Volume & Frequency Features
# Focus on purchases for spend-related features
purchases = df[df['tipo_operacion'] == 'compra'].copy()

# Add year-month for monthly aggregations
purchases['year_month'] = purchases['fecha_hora'].dt.to_period('M')

# Monthly spend sums per user
monthly_spend = purchases.groupby(['user_id', 'year_month'])['monto'].sum().reset_index()

# monthly_avg_spend: mean of monthly sums
monthly_avg_spend = monthly_spend.groupby('user_id')['monto'].mean().rename('monthly_avg_spend')

# txn_frequency: average monthly transaction count
monthly_txn_count = purchases.groupby(['user_id', 'year_month']).size().reset_index(name='count')
txn_frequency = monthly_txn_count.groupby('user_id')['count'].mean().rename('txn_frequency')

# avg_ticket_size: mean monto per purchase
avg_ticket_size = purchases.groupby('user_id')['monto'].mean().rename('avg_ticket_size')

# spend_volatility: coefficient of variation of monthly spends
# Only compute if at least 2 months of data
monthly_count = monthly_spend.groupby('user_id').size()
monthly_spend_stats = monthly_spend.groupby('user_id')['monto'].agg(['mean', 'std'])
monthly_spend_stats['spend_volatility'] = monthly_spend_stats['std'] / monthly_spend_stats['mean']
spend_volatility = monthly_spend_stats['spend_volatility'].where(monthly_count >= 2, 0).fillna(0)

# failed_txn_rate: proportion of failed transactions
failed_txn_rate = df.groupby('user_id')['estatus'].apply(lambda x: (x == 'no_procesada').mean()).rename('failed_txn_rate')

# retry_rate: proportion with intento_numero > 1
retry_rate = df.groupby('user_id')['intento_numero'].apply(lambda x: (x > 1).mean()).rename('retry_rate')

# dispute_rate: proportion with estatus == 'en_disputa'
dispute_rate = df.groupby('user_id')['estatus'].apply(lambda x: (x == 'en_disputa').mean()).rename('dispute_rate')

# Combine volume & frequency features
volume_freq_features = pd.concat([monthly_avg_spend, txn_frequency, avg_ticket_size, spend_volatility, failed_txn_rate, retry_rate, dispute_rate], axis=1).fillna(0)
print("Volume & Frequency features shape:", volume_freq_features.shape)
print(volume_freq_features.head())

Volume & Frequency features shape: (15025, 7)
           monthly_avg_spend  txn_frequency  avg_ticket_size  \
user_id                                                        
USR-00001        1215.000000       2.818182       431.129032   
USR-00002        1468.359091       3.909091       375.626744   
USR-00003        1638.176364       3.909091       419.068372   
USR-00004        5661.518182       2.454545      2306.544444   
USR-00005        1534.595455       4.181818       366.968478   

           spend_volatility  failed_txn_rate  retry_rate  dispute_rate  
user_id                                                                 
USR-00001          0.579754         0.053571    0.035714      0.000000  
USR-00002          0.587474         0.013514    0.013514      0.013514  
USR-00003          0.671421         0.050633    0.012658      0.037975  
USR-00004          0.720112         0.030303    0.015152      0.030303  
USR-00005          0.501935         0.021505    0.021505      0.021

## Spending Category Distribution (Lifestyle DNA)

In [134]:
# Spending Category Distribution
# Percentage of total spend per MCC category for purchases
category_spend = purchases.groupby(['user_id', 'categoria_mcc'])['monto'].sum().unstack(fill_value=0)

# Compute percentages
total_spend_per_user = category_spend.sum(axis=1)
lifestyle_dna = category_spend.div(total_spend_per_user, axis=0).fillna(0)

# Rename columns to pct_ prefixes
lifestyle_dna.columns = ['pct_' + col for col in lifestyle_dna.columns]

# Ensure all required categories are present (fill missing with 0)
required_cats = ['pct_supermercado', 'pct_restaurante', 'pct_entretenimiento', 'pct_viajes', 'pct_educacion', 'pct_salud', 'pct_tecnologia', 'pct_servicios_digitales', 'pct_ropa_accesorios', 'pct_transporte', 'pct_hogar', 'pct_gobierno', 'pct_transferencia', 'pct_delivery']
for cat in required_cats:
    if cat not in lifestyle_dna.columns:
        lifestyle_dna[cat] = 0

# Select only the specified features
selected_lifestyle = lifestyle_dna[['pct_supermercado', 'pct_restaurante', 'pct_entretenimiento', 'pct_viajes', 'pct_educacion', 'pct_salud', 'pct_tecnologia', 'pct_servicios_digitales', 'pct_ropa_accesorios', 'pct_transporte', 'pct_hogar']]

print("Lifestyle DNA features shape:", selected_lifestyle.shape)
print(selected_lifestyle.head())

Lifestyle DNA features shape: (14952, 11)
           pct_supermercado  pct_restaurante  pct_entretenimiento  pct_viajes  \
user_id                                                                         
USR-00001          0.062804         0.451993             0.227351    0.000000   
USR-00002          0.086723         0.498903             0.145168    0.000000   
USR-00003          0.030776         0.259852             0.159400    0.000000   
USR-00004          0.254392         0.155926             0.104638    0.165448   
USR-00005          0.071974         0.373844             0.349478    0.000000   

           pct_educacion  pct_salud  pct_tecnologia  pct_servicios_digitales  \
user_id                                                                        
USR-00001            0.0   0.000000        0.000000                 0.208067   
USR-00002            0.0   0.000000        0.025335                 0.106728   
USR-00003            0.0   0.000000        0.089951                 0.

## Credit Behavior Features

In [135]:
# Credit Behavior Features
# msi_usage_rate: proportion of purchases with MSI
msi_usage_rate = purchases.groupby('user_id')['meses_diferidos'].apply(lambda x: (x > 0).mean()).rename('msi_usage_rate')

# avg_msi_months: mean MSI months where used
avg_msi_months = purchases[purchases['meses_diferidos'] > 0].groupby('user_id')['meses_diferidos'].mean().rename('avg_msi_months')

# cashback_total: sum of cashback generated
cashback_total = df.groupby('user_id')['cashback_generado'].sum().rename('cashback_total')

# recurring_charge_count: count of recurring charges
recurring_charge_count = df[df['tipo_operacion'] == 'cargo_recurrente'].groupby('user_id').size().rename('recurring_charge_count')

# Combine credit behavior features
credit_behavior_features = pd.concat([msi_usage_rate, avg_msi_months, cashback_total, recurring_charge_count], axis=1).fillna(0)
print("Credit Behavior features shape:", credit_behavior_features.shape)
print(credit_behavior_features.head())

Credit Behavior features shape: (15025, 4)
           msi_usage_rate  avg_msi_months  cashback_total  \
user_id                                                     
USR-00001        0.000000             0.0          122.33   
USR-00002        0.000000             0.0          147.08   
USR-00003        0.000000             0.0          165.43   
USR-00004        0.148148             9.0            0.00   
USR-00005        0.000000             0.0          162.99   

           recurring_charge_count  
user_id                            
USR-00001                     9.0  
USR-00002                    12.0  
USR-00003                     8.0  
USR-00004                     5.0  
USR-00005                    17.0  


## Temporal Patterns Features

In [136]:
# Temporal Patterns Features
# night_owl_score: proportion of transactions between 22-5
night_hours = df['hora_del_dia'].isin(list(range(22, 24)) + list(range(0, 6)))
night_owl_score = df.groupby('user_id')['hora_del_dia'].apply(lambda x: night_hours.loc[x.index].mean()).rename('night_owl_score')

# weekend_spend_ratio: weekend spend / total spend
weekend_mask = df['dia_semana'].isin(['Saturday', 'Sunday'])
weekend_spend = df[weekend_mask].groupby('user_id')['monto'].sum()
total_spend = df.groupby('user_id')['monto'].sum()
weekend_spend_ratio = (weekend_spend / total_spend).fillna(0).rename('weekend_spend_ratio')

# peak_hour: mode of hora_del_dia
peak_hour = df.groupby('user_id')['hora_del_dia'].agg(lambda x: x.mode().iloc[0] if not x.mode().empty else 0).rename('peak_hour')

# recency_days: days since last completed transaction
completed_txns = df[df['estatus'] == 'completada']
recency_days = (ref_date - completed_txns.groupby('user_id')['fecha_hora'].max()).dt.days.rename('recency_days')

# Combine temporal features
temporal_features = pd.concat([night_owl_score, weekend_spend_ratio, peak_hour, recency_days], axis=1).fillna(0)
print("Temporal Patterns features shape:", temporal_features.shape)
print(temporal_features.head())

Temporal Patterns features shape: (15025, 4)
           night_owl_score  weekend_spend_ratio  peak_hour  recency_days
user_id                                                                 
USR-00001         0.303571             0.299770          1            23
USR-00002         0.270270             0.198637         19            18
USR-00003         0.265823             0.083239         17            20
USR-00004         0.287879             0.469787          7            19
USR-00005         0.387097             0.366191          6            29


## Geography & Internationality Features

In [137]:
# Geography & Internationality Features
# international_txn_rate: proportion of international transactions
international_txn_rate = df.groupby('user_id')['es_internacional'].mean().rename('international_txn_rate')

# city_diversity: number of unique cities
city_diversity = df.groupby('user_id')['ciudad_transaccion'].nunique().rename('city_diversity')

# Note: out_of_state_rate omitted as home state data not available

# Combine geography features
geography_features = pd.concat([international_txn_rate, city_diversity], axis=1).fillna(0)
print("Geography & Internationality features shape:", geography_features.shape)
print(geography_features.head())

Geography & Internationality features shape: (15025, 2)
           international_txn_rate  city_diversity
user_id                                          
USR-00001                0.035714               5
USR-00002                0.013514               6
USR-00003                0.050633               9
USR-00004                0.030303              12
USR-00005                0.010753               8


## Cash vs Digital Features

In [138]:
# Cash vs Digital Features
# cash_dependency: proportion of cash-related transactions
cash_types = ['retiro_cajero', 'deposito_oxxo', 'deposito_farmacia']
cash_dependency = df.groupby('user_id')['tipo_operacion'].apply(lambda x: x.isin(cash_types).mean()).rename('cash_dependency')

# digital_payment_rate: proportion via digital channels
digital_channels = ['app_ios', 'app_android', 'app_huawei', 'codi']
digital_payment_rate = df.groupby('user_id')['canal'].apply(lambda x: x.isin(digital_channels).mean()).rename('digital_payment_rate')

# Combine cash vs digital features
cash_digital_features = pd.concat([cash_dependency, digital_payment_rate], axis=1).fillna(0)
print("Cash vs Digital features shape:", cash_digital_features.shape)
print(cash_digital_features.head())

Cash vs Digital features shape: (15025, 2)
           cash_dependency  digital_payment_rate
user_id                                         
USR-00001         0.035714              0.839286
USR-00002         0.027027              0.716216
USR-00003         0.050633              0.670886
USR-00004         0.015152              0.848485
USR-00005         0.064516              0.709677


## Risk/Anomaly Signals Features

In [139]:
# Risk/Anomaly Signals Features
# atypical_txn_rate: proportion of atypical transactions
atypical_txn_rate = df.groupby('user_id')['patron_uso_atipico'].mean().rename('atypical_txn_rate')

# large_txn_count: count of transactions above user's 95th percentile
def count_large_txns(group):
    if len(group) == 0:
        return 0
    pct95 = group['monto'].quantile(0.95)
    return (group['monto'] > pct95).sum()

large_txn_count = df.groupby('user_id').apply(count_large_txns).rename('large_txn_count')

# Combine risk features
risk_features = pd.concat([atypical_txn_rate, large_txn_count], axis=1).fillna(0)
print("Risk/Anomaly Signals features shape:", risk_features.shape)
print(risk_features.head())

Risk/Anomaly Signals features shape: (15025, 2)
           atypical_txn_rate  large_txn_count
user_id                                      
USR-00001                0.0                3
USR-00002                0.0                4
USR-00003                0.0                4
USR-00004                0.0                4
USR-00005                0.0                5


/tmp/ipykernel_18389/3158897664.py:12: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  large_txn_count = df.groupby('user_id').apply(count_large_txns).rename('large_txn_count')


## Merge the categories

In [140]:
# Merge all features into one dataframe
transactions_final = pd.concat([
    volume_freq_features,
    selected_lifestyle,
    credit_behavior_features,
    temporal_features,
    geography_features,
    cash_digital_features,
    risk_features
], axis=1, join='outer').fillna(0)

# Reset index to have user_id as column
transactions_final = transactions_final.reset_index()

print("Final features shape:", transactions_final.shape)
print("Columns:", list(transactions_final.columns))
print(transactions_final.head())

Final features shape: (15025, 33)
Columns: ['user_id', 'monthly_avg_spend', 'txn_frequency', 'avg_ticket_size', 'spend_volatility', 'failed_txn_rate', 'retry_rate', 'dispute_rate', 'pct_supermercado', 'pct_restaurante', 'pct_entretenimiento', 'pct_viajes', 'pct_educacion', 'pct_salud', 'pct_tecnologia', 'pct_servicios_digitales', 'pct_ropa_accesorios', 'pct_transporte', 'pct_hogar', 'msi_usage_rate', 'avg_msi_months', 'cashback_total', 'recurring_charge_count', 'night_owl_score', 'weekend_spend_ratio', 'peak_hour', 'recency_days', 'international_txn_rate', 'city_diversity', 'cash_dependency', 'digital_payment_rate', 'atypical_txn_rate', 'large_txn_count']
     user_id  monthly_avg_spend  txn_frequency  avg_ticket_size  \
0  USR-00001        1215.000000       2.818182       431.129032   
1  USR-00002        1468.359091       3.909091       375.626744   
2  USR-00003        1638.176364       3.909091       419.068372   
3  USR-00004        5661.518182       2.454545      2306.544444   
4

# Final merge

In [141]:
# Load original clients data to retain raw features
df_clients = pd.read_csv('../../Data/dataset_transacciones/hey_clientes.csv')

# Merge with engineered client features
merged = df_clients.merge(clients_final, on='user_id', how='left')

# Merge with products features
merged = merged.merge(products_final, on='user_id', how='left')

# Merge with transactions features
merged = merged.merge(transactions_final, on='user_id', how='left')

# Save the complete merged dataframe
merged.to_csv('hey_banco_complete_features.csv', index=False)

print("Merged dataframe shape:", merged.shape)
print("Columns:", list(merged.columns))
print(merged.head())

Merged dataframe shape: (15025, 71)
Columns: ['user_id', 'edad', 'sexo', 'estado', 'ciudad', 'nivel_educativo', 'ocupacion', 'ingreso_mensual_mxn', 'antiguedad_dias', 'es_hey_pro_x', 'nomina_domiciliada', 'canal_apertura', 'score_buro', 'dias_desde_ultimo_login', 'preferencia_canal', 'satisfaccion_1_10', 'recibe_remesas', 'usa_hey_shop', 'idioma_preferido', 'tiene_seguro', 'num_productos_activos', 'patron_uso_atipico', 'age_band', 'income_tier', 'es_hey_pro_y', 'engagement_score', 'digital_maturity', 'vulnerability_flag', 'financial_sophistication', 'product_diversity_score', 'has_credit', 'avg_credit_utilization', 'max_credit_limit', 'has_investment', 'has_insurance', 'secured_card_flag', 'portfolio_age_days', 'credit_product_count', 'investment_balance', 'monthly_avg_spend', 'txn_frequency', 'avg_ticket_size', 'spend_volatility', 'failed_txn_rate', 'retry_rate', 'dispute_rate', 'pct_supermercado', 'pct_restaurante', 'pct_entretenimiento', 'pct_viajes', 'pct_educacion', 'pct_salud', '